In [1]:
import pandas as pd
import os


In [5]:
a=pd.read_csv("summary_table.csv")
a

,file,avg_source_probability,avg_destination_probability,most_probable_source,most_probable_destination
0,g10_ignore_bimodal_exponential_filtered_10.csv,0.013514,0.500000,02ec20f34bb94460f3d63780dfc24a4d4a1ddabc3bd86c...,03de68d2d4ced8dd71f0e245b96dc0079823941becad01...
1,g10_ignore_bimodal_exponential_filtered_100.csv,0.500000,0.003817,02e8ff7cf72482fec25e7d07f2d1c0f880ec4d0e372da0...,0372aa928f48eb664eded0773829c1dcd7fccfbda81e1b...
2,g10_ignore_bimodal_exponential_filtered_1000.csv,0.035714,0.050000,0364913d18a19c671bb36dd04d6ad5be0fe8f2894314c3...,0355a43ef6fc181bbc5146fa8e5926b91cd7adbb3ce192...
3,g10_ignore_bimodal_exponential_filtered_10000.csv,0.500000,0.019608,0352eb176b4c40cb195e8ffc009caacfe0d02ae0c0379f...,02eef44bb2092b9327d7e16cca6ece40b9d35a64301b3c...
4,g10_ignore_bimodal_uniform_filtered_10.csv,0.111111,0.250000,02c095d069538f96bf14c5f90f6c0851bdf354a0ec8603...,033fc997c1622281b143fce0cce0711f18fdc812968484...
5,g10_ignore_bimodal_uniform_filtered_100.csv,0.010417,0.015625,03214d3213d6da40265099466f500d8f4812ce2c23af02...,02c376e54ff08c4ab3c94e9a5cebdef2c367d29a428b9f...
6,g10_ignore_bimodal_uniform_filtered_1000.csv,0.500000,0.005405,03f576c4fe7ec82a459dbc194b33b55f4e9a606442f693...,037f6c805dfef7a0f01cca0abcea96352a32aa478c63c9...
7,g10_ignore_bimodal_uniform_filtered_10000.csv,0.012195,0.011236,02cc79b8d771cc79d323bd71e54ef1780cab50e7c0ef2a...,0217890e3aad8d35bc054f43acc00084b25229ecff0ab6...
8,g10_ignore_filtered_10.csv,0.018182,0.500000,030c3f19d742ca294a55c00376b3b355c3c90d61c6b6b3...,032d9f76e3fcc736734a0fae90fe88218f690c0e9e2d5c...
9,g10_ignore_filtered_100.csv,0.008929,0.250000,0340cfadaa3324e0dd176a9969be050114278f93260e1b...,021c26488cc56c17fb81938b95e680463ed9cc352243b4...


In [1]:
import pandas as pd
import re

# Load summary table
df = pd.read_csv("summary_table.csv")
df.columns = df.columns.str.strip()

# Choose group: "ignore" or "merge"
group = "ignore"
df = df[df['file'].str.contains(group)]

# Parse distribution and amount
def parse_file_info(fname):
    if "uniform" in fname:
        dist = "Uniform"
    elif "bimodal" in fname:
        dist = "Bimodal"
    else:
        dist = "Normal"
    amount_match = re.search(r'_([0-9]+)\.csv$', fname)
    amount = int(amount_match.group(1)) if amount_match else 0
    return dist, amount

df[["distribution", "amount"]] = df["file"].apply(lambda x: pd.Series(parse_file_info(x)))

# Sort order
dist_order = ["Normal", "Uniform", "Bimodal"]
df["dist_order"] = df["distribution"].apply(lambda x: dist_order.index(x))
df = df.sort_values(["amount", "dist_order"])
amounts = sorted(df["amount"].unique())

def print_table(prob_col, title):
    print(r"\begin{table}[h!]")
    print(r"\centering")
    print(r"\begin{tabular}{|c|c|c|c|}")
    print(r"\hline")
    print(r"Amount & Normal & Uniform & Bimodal \\")
    print(r"\hline")
    
    for amt in amounts:
        row = [str(amt)]
        for dist in dist_order:
            sub = df[(df["amount"] == amt) & (df["distribution"] == dist)]
            if not sub.empty:
                val = sub[prob_col].values[0]
                row.append(f"{val:.4f}")
            else:
                row.append("")
        print(" & ".join(row) + r" \\")
    
    print(r"\hline")
    print(r"\end{tabular}")
    print(rf"\caption{{Average {title} Probability of the nodes in the set}}")
    print(r"\end{table}")
    print()

# First table → Source probabilities
print_table("avg_source_probability", "Source")

# Second table → Destination probabilities
print_table("avg_destination_probability", "Destination")


\begin{table}[h!]
\centering
\begin{tabular}{|c|c|c|c|}
\hline
Amount & Normal & Uniform & Bimodal \\
\hline
10 & 0.0182 & 0.1111 & 0.0135 \\
100 & 0.0089 & 0.0104 & 0.5000 \\
1000 & 0.5000 & 0.5000 & 0.0357 \\
10000 & 0.5000 & 0.0122 & 0.5000 \\
\hline
\end{tabular}
\caption{Average Source Probability of the nodes in the set}
\end{table}

\begin{table}[h!]
\centering
\begin{tabular}{|c|c|c|c|}
\hline
Amount & Normal & Uniform & Bimodal \\
\hline
10 & 0.5000 & 0.2500 & 0.5000 \\
100 & 0.2500 & 0.0156 & 0.0038 \\
1000 & 0.0014 & 0.0054 & 0.0500 \\
10000 & 0.0004 & 0.0112 & 0.0196 \\
\hline
\end{tabular}
\caption{Average Destination Probability of the nodes in the set}
\end{table}

